> Paths here point at a flat working folder (`generated_sequences/`, `dna/`,
> `pdbqt/`, `output/`, `template/`), not at this repo's layout. The folders under
> `data/` use the same names, so copy them into one folder to re-run. See
> `docs/RERUN.md`.
>
> Code cells and stored outputs are as they were run. Only markdown was fixed
> where it contradicted the code.


# Load library and path

In [1]:
import os
import numpy as np
import subprocess

# Base directory (portable)
base_dir = os.getcwd()
print("Working directory:", base_dir)

# Folders
dna_dir = os.path.join(base_dir, "dna")
pdbqt_dir = os.path.join(base_dir, "pdbqt")
output_dir = os.path.join(base_dir, "output")
template_dir = os.path.join(base_dir, "template")

# Files
vina_exe = os.path.join(base_dir, "vina.exe")
ligand_pdb = os.path.join(template_dir, "template.pdb")
ligand_pdbqt = os.path.join(template_dir, "template.pdbqt")

Working directory: C:\IQS-DNA2


# Check environment

In [2]:
print("DNA files:", os.listdir(dna_dir))
print("PDBQT folder:", os.listdir(pdbqt_dir))
print("Ligand PDB exists:", os.path.exists(ligand_pdb))
print("Vina exists:", os.path.exists(vina_exe))

DNA files: ['model (1).pdb', 'model (10).pdb', 'model (100).pdb', 'model (11).pdb', 'model (12).pdb', 'model (13).pdb', 'model (14).pdb', 'model (15).pdb', 'model (16).pdb', 'model (17).pdb', 'model (18).pdb', 'model (19).pdb', 'model (2).pdb', 'model (20).pdb', 'model (21).pdb', 'model (22).pdb', 'model (23).pdb', 'model (24).pdb', 'model (25).pdb', 'model (26).pdb', 'model (27).pdb', 'model (28).pdb', 'model (29).pdb', 'model (3).pdb', 'model (30).pdb', 'model (31).pdb', 'model (32).pdb', 'model (33).pdb', 'model (34).pdb', 'model (35).pdb', 'model (36).pdb', 'model (37).pdb', 'model (38).pdb', 'model (39).pdb', 'model (4).pdb', 'model (40).pdb', 'model (41).pdb', 'model (42).pdb', 'model (43).pdb', 'model (44).pdb', 'model (45).pdb', 'model (46).pdb', 'model (47).pdb', 'model (48).pdb', 'model (49).pdb', 'model (5).pdb', 'model (50).pdb', 'model (51).pdb', 'model (52).pdb', 'model (53).pdb', 'model (54).pdb', 'model (55).pdb', 'model (56).pdb', 'model (57).pdb', 'model (58).pdb', 'm

# Convert ligand / template into PDBQT

In [3]:
if not os.path.exists(ligand_pdbqt):
    print("🔄 Converting ligand to PDBQT...")
    os.system(f'obabel "{ligand_pdb}" -O "{ligand_pdbqt}" --partialcharge gasteiger')
    print(f"✅ Done")
else:
    print("✅ Ligand already converted")

✅ Ligand already converted


# Convert all DNA into PDBQT

In [4]:
for file in os.listdir(dna_dir):
    if file.endswith(".pdb") or file.endswith(".PDB"):
        
        name = os.path.splitext(file)[0]
        input_path = os.path.join(dna_dir, file)
        output_path = os.path.join(pdbqt_dir, name + ".pdbqt")
        
        if not os.path.exists(output_path):
            print(f"🔄 Converting {name} ✅ Done!")
            cmd = f'obabel "{input_path}" -O "{output_path}" --partialcharge gasteiger'
            os.system(cmd)
        else:
            print(f"✅ Already exists: {name}")

🔄 Converting model (1) ✅ Done!
🔄 Converting model (10) ✅ Done!
🔄 Converting model (100) ✅ Done!
🔄 Converting model (11) ✅ Done!
🔄 Converting model (12) ✅ Done!
🔄 Converting model (13) ✅ Done!
🔄 Converting model (14) ✅ Done!
🔄 Converting model (15) ✅ Done!
🔄 Converting model (16) ✅ Done!
🔄 Converting model (17) ✅ Done!
🔄 Converting model (18) ✅ Done!
🔄 Converting model (19) ✅ Done!
🔄 Converting model (2) ✅ Done!
🔄 Converting model (20) ✅ Done!
🔄 Converting model (21) ✅ Done!
🔄 Converting model (22) ✅ Done!
🔄 Converting model (23) ✅ Done!
🔄 Converting model (24) ✅ Done!
🔄 Converting model (25) ✅ Done!
🔄 Converting model (26) ✅ Done!
🔄 Converting model (27) ✅ Done!
🔄 Converting model (28) ✅ Done!
🔄 Converting model (29) ✅ Done!
🔄 Converting model (3) ✅ Done!
🔄 Converting model (30) ✅ Done!
🔄 Converting model (31) ✅ Done!
🔄 Converting model (32) ✅ Done!
🔄 Converting model (33) ✅ Done!
🔄 Converting model (34) ✅ Done!
🔄 Converting model (35) ✅ Done!
🔄 Converting model (36) ✅ Done!
🔄 Converti

# Compute center function

In [5]:
def get_center(pdb_file):
    coords = []
    with open(pdb_file, "r") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])
    coords = np.array(coords)
    return coords.mean(axis=0)

print(f"✅ Finished function to get center point!")

✅ Finished function to get center point!


# Fix all the DNA PBDQT files

In [6]:
def fix_receptor_pdbqt(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()
    
    cleaned = []
    for line in lines:
        if line.startswith(("ROOT", "ENDROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            continue
        cleaned.append(line)
    
    with open(file_path, "w") as f:
        f.writelines(cleaned)

In [7]:
for file in os.listdir(pdbqt_dir):
    if file.endswith(".pdbqt"):
        path = os.path.join(pdbqt_dir, file)
        fix_receptor_pdbqt(path)
        print("Fixed:", file)

Fixed: model (1).pdbqt
Fixed: model (10).pdbqt
Fixed: model (100).pdbqt
Fixed: model (11).pdbqt
Fixed: model (12).pdbqt
Fixed: model (13).pdbqt
Fixed: model (14).pdbqt
Fixed: model (15).pdbqt
Fixed: model (16).pdbqt
Fixed: model (17).pdbqt
Fixed: model (18).pdbqt
Fixed: model (19).pdbqt
Fixed: model (2).pdbqt
Fixed: model (20).pdbqt
Fixed: model (21).pdbqt
Fixed: model (22).pdbqt
Fixed: model (23).pdbqt
Fixed: model (24).pdbqt
Fixed: model (25).pdbqt
Fixed: model (26).pdbqt
Fixed: model (27).pdbqt
Fixed: model (28).pdbqt
Fixed: model (29).pdbqt
Fixed: model (3).pdbqt
Fixed: model (30).pdbqt
Fixed: model (31).pdbqt
Fixed: model (32).pdbqt
Fixed: model (33).pdbqt
Fixed: model (34).pdbqt
Fixed: model (35).pdbqt
Fixed: model (36).pdbqt
Fixed: model (37).pdbqt
Fixed: model (38).pdbqt
Fixed: model (39).pdbqt
Fixed: model (4).pdbqt
Fixed: model (40).pdbqt
Fixed: model (41).pdbqt
Fixed: model (42).pdbqt
Fixed: model (43).pdbqt
Fixed: model (44).pdbqt
Fixed: model (45).pdbqt
Fixed: model (46).p

## Test seq1

# Run all other sequences

In [8]:
import os
print(os.cpu_count())

16


In [9]:
results = []
size = [40, 40, 40]

for file in os.listdir(dna_dir):
    if file.endswith(".pdb"):
        
        name = file.replace(".pdb", "")
        
        pdb_path = os.path.join(dna_dir, file)
        pdbqt_path = os.path.join(pdbqt_dir, name + ".pdbqt")
        out_path = os.path.join(output_dir, name + "_out.pdbqt")
        
        if not os.path.exists(pdbqt_path):
            print("Missing:", name)
            continue
        
        center = get_center(pdb_path)
        cx, cy, cz = center
        
        print("Running:", name)  # short progress only
        
        cmd = [
            vina_exe,
            "--receptor", pdbqt_path,
            "--ligand", ligand_pdbqt,
            "--center_x", str(cx),
            "--center_y", str(cy),
            "--center_z", str(cz),
            "--size_x", str(size[0]),
            "--size_y", str(size[1]),
            "--size_z", str(size[2]),
            "--exhaustiveness", "32",
            "--num_modes", "30",
            "--cpu", "12",
            "--out", out_path
        ]
        
        # 👇 suppress all output
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

Running: model (1)
Running: model (10)
Running: model (100)
Running: model (11)
Running: model (12)
Running: model (13)
Running: model (14)
Running: model (15)
Running: model (16)
Running: model (17)
Running: model (18)
Running: model (19)
Running: model (2)
Running: model (20)
Running: model (21)
Running: model (22)
Running: model (23)
Running: model (24)
Running: model (25)
Running: model (26)
Running: model (27)
Running: model (28)
Running: model (29)
Running: model (3)
Running: model (30)
Running: model (31)
Running: model (32)
Running: model (33)
Running: model (34)
Running: model (35)
Running: model (36)
Running: model (37)
Running: model (38)
Running: model (39)
Running: model (4)
Running: model (40)
Running: model (41)
Running: model (42)
Running: model (43)
Running: model (44)
Running: model (45)
Running: model (46)
Running: model (47)
Running: model (48)
Running: model (49)
Running: model (5)
Running: model (50)
Running: model (51)
Running: model (52)
Running: model (53)
Runn

In [10]:
results = []

for file in os.listdir(output_dir):
    if file.endswith("_out.pdbqt"):
        
        name = file.replace("_out.pdbqt", "")
        path = os.path.join(output_dir, file)
        
        with open(path, "r") as f:
            for line in f:
                if "REMARK VINA RESULT" in line:
                    score = float(line.split()[3])
                    results.append((name, score))
                    break

results.sort(key=lambda x: x[1])

print("\n=== FINAL RANKING ===")
for r in results:
    print(r)


=== FINAL RANKING ===
('model (87)', -6.934)
('model (59)', -6.931)
('model (23)', -6.898)
('model (49)', -6.879)
('model (56)', -6.875)
('model (81)', -6.874)
('model (72)', -6.871)
('model (52)', -6.866)
('model (60)', -6.839)
('model (77)', -6.819)
('model (74)', -6.813)
('model (58)', -6.774)
('model (40)', -6.767)
('model (64)', -6.766)
('model (16)', -6.757)
('model (68)', -6.755)
('model (89)', -6.752)
('model (76)', -6.751)
('model (46)', -6.75)
('model (96)', -6.75)
('model (25)', -6.749)
('model (98)', -6.747)
('model (35)', -6.744)
('model (82)', -6.741)
('model (85)', -6.741)
('model (53)', -6.74)
('model (6)', -6.736)
('model (80)', -6.735)
('model (22)', -6.734)
('model (90)', -6.732)
('model (31)', -6.731)
('model (86)', -6.731)
('model (57)', -6.726)
('model (19)', -6.725)
('model (28)', -6.721)
('model (70)', -6.72)
('model (100)', -6.718)
('model (29)', -6.718)
('model (18)', -6.705)
('model (26)', -6.704)
('model (95)', -6.696)
('model (99)', -6.688)
('model (92)', 

In [11]:
# --- Save Results ---

txt_path = os.path.join(base_dir, "docking_results.txt")

with open(txt_path, "w") as f:
    f.write("=== FINAL DOCKING RANKING ===\n")
    f.write(f"{'Rank':<6} {'Model':<20} {'Binding Affinity (kcal/mol)'}\n")
    f.write("-" * 45 + "\n")
    for rank, (name, score) in enumerate(results, start=1):
        f.write(f"{rank:<6} {name:<20} {score}\n")

print(f"TXT saved: {txt_path}")

TXT saved: C:\IQS-DNA2\docking_results.txt
